# Poisoning Budget Analysis (Intermediate)

This notebook studies **how the proportion of poisoned training data** affects classifier performance under a **white-box, training-time label-flipping** attack on two text-based tasks:

1. **Binary classification** — Enron spam detection  
2. **Multi-class classification** — UMUDGA malware domain classification

---

## Theoretical framework

### Attack model: random label-flipping

The attacker does **not** modify features $(X)$. They only corrupt a subset of training labels $(y)$ in the pristine training set. This is a classic **availability / reliability** poisoning pattern: the model is trained on misleading supervision while inputs remain realistic.

### Cost budget \(C\)

$(C \in [0, 1])$ is the **maximum fraction of training points** the attacker may relabel. For example, $(C = 0.10)$ means at most 10% of training labels are flipped. Larger $(C)$ means a more expensive attack (more annotations corrupted).

### Attacker goal: reliability attack

The objective is to **maximize prediction error on clean test data** — i.e., degrade generalization on held-out, unmodified examples. I therefore keep the **test set completely clean** and only poison **training** labels.



## Step 1: Setup and data loading


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Reproducibility
RANDOM_STATE = 40
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
ENRON_CSV = DATA_DIR / "enron_spam.csv"
UMUDGA_CSV = DATA_DIR / "umudga.csv"

POISONING_BUDGETS = [0.0, 0.01, 0.05, 0.10, 0.15, 0.20, 0.30]
MAX_TFIDF_FEATURES = 5000
TEST_SIZE = 0.20

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
def load_enron_dataset(csv_path: Path = ENRON_CSV) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required = {"text", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Enron CSV missing columns {missing}. "
            "Rename columns to 'text' and 'label' in this function."
        )
    return df[["text", "label"]].dropna().reset_index(drop=True)


def load_umudga_dataset(csv_path: Path = UMUDGA_CSV) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required = {"text", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"UMUDGA CSV missing columns {missing}. "
            "Rename columns to 'text' and 'label' in this function."
        )
    return df[["text", "label"]].dropna().reset_index(drop=True)

## Step 2: Preprocessing and feature extraction

Text is converted with **TF–IDF** (`max_features=5000` for speed). I use an **80/20 train–test split**. The **test labels and features are never poisoned** — they are the ground truth for measuring attacker utility (degradation on clean data).

In [ ]:
def preprocess_and_vectorize(
    df: pd.DataFrame,
    max_features: int = MAX_TFIDF_FEATURES,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    """TF-IDF features, label encoding, and stratified train/test split."""
    texts = df["text"].astype(str)
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df["label"])
    num_classes = len(label_encoder.classes_)

    vectorizer = TfidfVectorizer(max_features=max_features)
    X = vectorizer.fit_transform(texts)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )
    return X_train, X_test, y_train, y_test, num_classes, label_encoder

## Step 3: Poisoning logic (label flipping)

Given budget $(C)$, I uniformly sample $(\lfloor C \cdot n \rfloor)$ training indices (at least one when $(C > 0)$ and $(n \geq 1))$ and flip their labels:

- **Binary:** $(0 \leftrightarrow 1)$  
- **Multi-class:** replace with a **uniformly random incorrect** class

Features $(X_{\text{train}})$ are unchanged — only $(y_{\text{train}})$ is corrupted, respecting the budget constraint.

In [ ]:
def apply_label_flipping(
    y_train: np.ndarray,
    budget: float,
    num_classes: int,
    random_state: int = RANDOM_STATE,
) -> np.ndarray:
    y_poisoned = np.array(y_train, copy=True)
    n = len(y_poisoned)
    if budget <= 0 or n == 0:
        return y_poisoned

    n_poison = int(np.floor(budget * n))
    if n_poison < 1:
        n_poison = 1
    n_poison = min(n_poison, n)

    rng = np.random.default_rng(random_state)
    poison_indices = rng.choice(n, size=n_poison, replace=False)

    for idx in poison_indices:
        original = y_poisoned[idx]
        if num_classes == 2:
            y_poisoned[idx] = 1 - original
        else:
            candidates = [c for c in range(num_classes) if c != original]
            y_poisoned[idx] = rng.choice(candidates)

    return y_poisoned

## Step 4: Experimental loop

For each dataset and each budget $(C \in \{0, 1\%, 5\%, \ldots, 30\%\})$:

1. Flip labels on the training set only.  
2. Retrain **Logistic Regression** (fast linear baseline on sparse TF–IDF).  
3. Evaluate on the **clean test set**.  
4. Record Accuracy, Precision, Recall, and F1 (macro for multi-class).

Rising $(C)$ should monotonically stress the model under a reliability attack, manifesting as loIr attacker-facing utility (worse metrics for the defender).

In [ ]:
def build_classifier(random_state: int = RANDOM_STATE) -> LogisticRegression:
    """Fast linear baseline for sparse TF-IDF features."""
    return LogisticRegression(
        max_iter=1000,
        solver="lbfgs",
        random_state=random_state,
    )


def evaluate_on_clean_test(
    y_test: np.ndarray,
    y_pred: np.ndarray,
    average: str,
) -> dict:
    """Compute standard metrics on unpoisoned test labels."""
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_test, y_pred, average=average, zero_division=0),
        "f1": f1_score(y_test, y_pred, average=average, zero_division=0),
    }


def run_poisoning_experiment(
    X_train,
    X_test,
    y_train: np.ndarray,
    y_test: np.ndarray,
    num_classes: int,
    dataset_name: str,
    budgets: list = None,
    metric_average: str = "binary",
) -> pd.DataFrame:
    """SIep poisoning budgets; return one row per (budget, metric)."""
    if budgets is None:
        budgets = POISONING_BUDGETS

    rows = []
    for budget in budgets:
        y_train_poisoned = apply_label_flipping(
            y_train, budget=budget, num_classes=num_classes
        )
        clf = build_classifier()
        clf.fit(X_train, y_train_poisoned)
        y_pred = clf.predict(X_test)
        scores = evaluate_on_clean_test(y_test, y_pred, average=metric_average)

        for metric_name, value in scores.items():
            rows.append(
                {
                    "dataset": dataset_name,
                    "budget": budget,
                    "budget_pct": budget * 100,
                    "metric": metric_name,
                    "score": value,
                }
            )
    return pd.DataFrame(rows)

### Run: Enron (binary) and UMUDGA (multi-class)

In [ ]:
enron_df = load_enron_dataset()
umudga_df = load_umudga_dataset()

print(f"Enron samples: {len(enron_df):,} | classes: {enron_df['label'].nunique()}")
print(f"UMUDGA samples: {len(umudga_df):,} | classes: {umudga_df['label'].nunique()}")

In [ ]:
X_train_e, X_test_e, y_train_e, y_test_e, n_classes_e, _ = preprocess_and_vectorize(enron_df)
X_train_u, X_test_u, y_train_u, y_test_u, n_classes_u, le_u = preprocess_and_vectorize(umudga_df)

results_enron = run_poisoning_experiment(
    X_train_e,
    X_test_e,
    y_train_e,
    y_test_e,
    num_classes=n_classes_e,
    dataset_name="Enron (Binary)",
    metric_average="binary",
)

results_umudga = run_poisoning_experiment(
    X_train_u,
    X_test_u,
    y_train_u,
    y_test_u,
    num_classes=n_classes_u,
    dataset_name="UMUDGA (Multi-Class)",
    metric_average="macro",
)

results_all = pd.concat([results_enron, results_umudga], ignore_index=True)
results_all

## Step 5: Evaluation and visualization

Pivot tables summarize attacker utility: as **Poisoning Budget (%)** grows, metric scores on the **clean test set** should fall, indicating a successful reliability attack within budget $(C)$.

In [ ]:
def results_to_wide(df: pd.DataFrame) -> pd.DataFrame:
    """Wide table: one row per budget, columns per metric."""
    wide = df.pivot_table(
        index=["dataset", "budget_pct"],
        columns="metric",
        values="score",
    ).reset_index()
    wide.columns.name = None
    return wide.sort_values("budget_pct")


print("=== Enron (Binary) ===")
display(results_to_wide(results_enron))

print("\n=== UMUDGA (Multi-Class) ===")
display(results_to_wide(results_umudga))

In [ ]:
def plot_poisoning_degradation(
    df: pd.DataFrame,
    title: str,
    save_path=None,
) -> None:
    """Line plot: Poisoning Budget (%) vs metric scores."""
    fig, ax = plt.subplots(figsize=(10, 6))
    metrics_order = ["accuracy", "precision", "recall", "f1"]
    palette = sns.color_palette("tab10", n_colors=len(metrics_order))

    for metric, color in zip(metrics_order, palette):
        subset = df[df["metric"] == metric].sort_values("budget_pct")
        ax.plot(
            subset["budget_pct"],
            subset["score"],
            marker="o",
            linewidth=2,
            label=metric.capitalize(),
            color=color,
        )

    ax.set_xlabel("Poisoning Budget (%)")
    ax.set_ylabel("Metric Score")
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.legend(title="Metric", loc="best")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


plot_poisoning_degradation(
    results_enron,
    title="Label-Flipping Attack: Enron Spam (Binary Classification)",
    save_path=Path("enron_poisoning_budget.png"),
)

plot_poisoning_degradation(
    results_umudga,
    title="Label-Flipping Attack: UMUDGA Malware Domains (Multi-Class)",
    save_path=Path("umudga_poisoning_budget.png"),
)